# Step 02: Detect and Audit Dataset Anomalies
### Automated 8-Rule Quality Inspector, Multi-Panel Diagnostic Graphs & Visual Sample Audits
Directly integrated with `multi_class_train_rfdetr.ipynb`'s Step B verification (Cell 12).

### 8 Anomaly Detection Rules:
1. **Degenerate Boxes**: Width or height <= 0.
2. **Micro-Clicks**: Accidental clicks with width/height < 4px or area < 16px.
3. **Giant Boxes**: Oversized bounding boxes covering > 95% of image area.
4. **Severe Image Blur**: Blurry images detected via Laplacian variance threshold (< 80.0).
5. **Edge-Cropped / Truncated Boxes**: Bounding boxes abruptly cropped by image borders (touching x=0, y=0, x+w>=W, or y+h>=H).
6. **Exact Duplicates**: Overlapping boxes of the same class with IoU > 0.95.
7. **Loose Margin Padding**: Excessive background padding around tags.
8. **Dense Query Overlaps**: Overly clustered overlapping tags.

### Key Outputs:
- Multi-Panel Anomaly Diagnostic Graphs (Distribution, Aspect Ratios, Areas, Density).
- Visual Sample Inspector displaying flagged anomalies (red boxes & yellow reason tags) vs clean boxes (green).
- `anomalies_manifest.json`: Complete record of all flagged anomaly boxes.
- `cleaned_dataset/`: Sanitized COCO annotations strictly excluding all anomalies.

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
print_vram_usage("Init")
logger.info("=" * 65)


import io
def show_plt_figure(fig=None):
    """Guarantees inline plot display in Jupyter under headless and Agg backends."""
    if fig is None:
        fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    pil_img = Image.open(buf)
    plt.close(fig)
    display(pil_img)

In [ ]:
# CELL 2 - Paths, Output Setup & Field Configuration
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

# Auto-discover active images pool directory across standard pipeline locations
candidate_image_dirs = [
    REPO_ROOT / "multi_class_train_rfdetr" / "images",
    INPUT_JSON_DIR / "images",
    REPO_ROOT / "images",
    Path("images")
]
IMAGES_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob("*"))), candidate_image_dirs[0])
os.makedirs(IMAGES_DIR, exist_ok=True)

# COCO annotation field mappings
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

# ==============================================================================
# MASTER ANOMALY AUDIT & SANITIZATION TOGGLES
# ==============================================================================
FILTER_EDGE_CROPPED_BOXES = False   # Set False: Border tags are legitimate valid objects (keep in training)
FILTER_MICRO_DEGENERATE = False     # Set False: Small/distant tags are legitimate valid objects (keep in training)
SHOW_ANOMALY_SAMPLES_PER_CATEGORY = 5   # Display exactly 5 sample images with tags for each anomaly category
MAX_DISPLAY_WIDTH = 750                 # Display width for inline notebook previews

OUTPUT_ANOMALIES_DIR = Path("anomalies_audit")
OUTPUT_ANOMALIES_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATASET_DIR = Path("cleaned_dataset")
CLEANED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Input JSON Dir:        {INPUT_JSON_DIR.resolve()}")
logger.info(f"Active Images Pool:    {IMAGES_DIR.resolve()} ({len(list(IMAGES_DIR.glob('*'))):,} cached images)")
logger.info(f"Anomalies Output Dir:  {OUTPUT_ANOMALIES_DIR.resolve()}")
logger.info(f"Cleaned Dataset Dir:   {CLEANED_DATASET_DIR.resolve()}")
logger.info(f"Show Samples Per Rule: {SHOW_ANOMALY_SAMPLES_PER_CATEGORY} sample images per anomaly category")


In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
raw_images = {}  # image_id / file_name -> {"id": ..., "width": ..., "height": ..., "file_name": ...}
raw_categories = {}  # category_id -> category_name

if len(json_files) <= 5:
    for jf in json_files:
        logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
else:
    logger.info(f"   - Ingesting {len(json_files)} JSON file(s)... (sample: {', '.join(p.name for p in json_files[:3])})")

for jf in tqdm(json_files, desc="[CELL 4] Reading JSON files", unit="file"):
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, dict):
                # Preserve full image metadata (width, height, file_name, id)
                for im in parsed.get("images", []):
                    if isinstance(im, dict):
                        iid = im.get("id")
                        if iid is not None:
                            raw_images[iid] = im
                            raw_images[str(iid)] = im
                        fn = im.get("file_name")
                        if fn:
                            raw_images[fn] = im
                            raw_images[Path(fn).name] = im
                            
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    for c in parsed.get("categories", []):
                        if isinstance(c, dict) and "id" in c and "name" in c:
                            raw_categories[c["id"]] = c["name"]
                            raw_categories[str(c["id"])] = c["name"]
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
            elif isinstance(parsed, list):
                raw_records.extend(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records):,} across {len(raw_images):,} registered image metadata entries.")


In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

# 1. Pre-scan for Coordinate System (Normalized [0..1] vs Absolute Pixels)
sample_bboxes = [
    item.get(BBOX_FIELD, []) for item in raw_records 
    if isinstance(item, dict) and isinstance(item.get(BBOX_FIELD), list) and len(item.get(BBOX_FIELD)) == 4
]

is_normalized_dataset = False
if sample_bboxes:
    max_sample_val = 0.0
    for b in sample_bboxes[:3000]:
        try:
            vals = [float(v) for v in b]
            if vals:
                max_sample_val = max(max_sample_val, max(vals))
        except (ValueError, TypeError):
            continue
            
    # If all coordinates in sample are <= 1.05, the dataset is in normalized [0, 1] format!
    if max_sample_val > 0.0 and max_sample_val <= 1.05:
        is_normalized_dataset = True
        logger.warning("=" * 70)
        logger.warning(f"⚠️ DETECTED NORMALIZED [0..1] BOUNDING BOXES! (sample max coordinate = {max_sample_val:.4f})")
        logger.warning("   Auto-scaling coordinates to absolute pixel space using image width/height.")
        logger.warning("=" * 70)

STANDARD_CLASSES = ["blue_aisle", "blue_bay", "location_tag"]

def resolve_target_category(item, raw_cat_map=None) -> str:
    c = item.get("category_name")
    if not c and "category_id" in item and raw_cat_map:
        cid = item["category_id"]
        c = raw_cat_map.get(cid, raw_cat_map.get(str(cid)))
    if not c:
        c = item.get("category") or item.get("category_id")
    if isinstance(c, list):
        c = c[0] if len(c) > 0 else ""
    c_str = str(c).strip().lower()
    if "aisle" in c_str or c_str in ["blue_aisle", "blue-aisle", "aisle"]:
        return "blue_aisle"
    elif "bay" in c_str or "blue_tag" in c_str or c_str in ["blue_bay", "bay"]:
        return "blue_bay"
    elif "location" in c_str or "white" in c_str or "loc" in c_str or c_str in ["location_tag", "white_tag", "tag", "default"]:
        return "location_tag"
    cid = item.get("category_id")
    if cid is not None:
        try:
            cid_int = int(cid)
            if cid_int in [0, 1, 2] and raw_cat_map and cid_int in raw_cat_map:
                mapped = str(raw_cat_map[cid_int]).lower()
                if "aisle" in mapped: return "blue_aisle"
                if "bay" in mapped or "blue_tag" in mapped: return "blue_bay"
                if "location" in mapped or "white" in mapped: return "location_tag"
            if cid_int == 0: return "blue_aisle"
            elif cid_int == 1: return "blue_bay"
            elif cid_int == 2: return "location_tag"
        except (ValueError, TypeError):
            pass
    return "location_tag"

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}
skipped_invalid_boxes = 0

for item in tqdm(raw_records, desc="[CELL 5] Parsing annotations", unit="rec"):
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    cat_name = resolve_target_category(item, raw_categories)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
    except (ValueError, TypeError):
        skipped_invalid_boxes += 1
        continue

    # Resolve true image dimensions from registered COCO metadata
    im_info = raw_images.get(img_id, raw_images.get(str(img_id), {}))
    img_w = float(im_info.get("width") or 1920)
    img_h = float(im_info.get("height") or 1080)

    # Scale normalized coordinates to absolute pixels if dataset is normalized
    if is_normalized_dataset or (x <= 1.05 and y <= 1.05 and w <= 1.05 and h <= 1.05 and (w > 0 or h > 0)):
        x = x * img_w
        y = y * img_h
        w = w * img_w
        h = h * img_h

    # Mathematically degenerate check (width or height must be positive)
    if w <= 0.0 or h <= 0.0:
        skipped_invalid_boxes += 1
        continue


    true_fn = (im_info.get("file_name") if im_info else None) or Path(str(item.get("image_url", f"{img_id}.jpg"))).name
    orig_url = (im_info.get("coco_url") if im_info else None) or item.get("image_url") or img_id
    ann_dict = {
        "id": item.get("id"),
        "image_id": img_id,
        "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", []),
        "img_w": img_w,
        "img_h": img_h,
        "file_name": true_fn,
        "image_url": orig_url
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

# Guarantee all 3 multi-class target categories
discovered_cats = set(category_counts.keys()).union(set(STANDARD_CLASSES))
for c in raw_categories.values():
    discovered_cats.add(resolve_target_category({"category_name": c}))
AUTO_CATEGORIES = sorted(list(discovered_cats))
if set(AUTO_CATEGORIES) == set(STANDARD_CLASSES):
    AUTO_CATEGORIES = list(STANDARD_CLASSES)
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]
        ann["image_id"] = img_id

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations):,}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes:,}")
if skipped_invalid_boxes > 0:
    logger.info(f"   - Skipped non-positive boxes:  {skipped_invalid_boxes:,} (w<=0 or h<=0)")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count:,} boxes, {count/max(1, total_boxes)*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes:,}")
# Construct definitions for downstream sanitized dataset exports
categories_list = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]
annotations_by_image = image_annotations
annotations_by_cat: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
for anns in image_annotations.values():
    for a in anns:
        annotations_by_cat[a["category_name"]].append(a)


In [ ]:
# CELL 5 - 8-Rule Automated High-Performance Anomaly Detector Engine
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# Master Toggle: Default treat ONLY blurry images as anomalies (user requirement)
TREAT_ONLY_BLUR_AS_ANOMALY = globals().get('TREAT_ONLY_BLUR_AS_ANOMALY', True)

def compute_box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    interArea = interW * interH
    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]
    denom = areaA + areaB - interArea
    return interArea / denom if denom > 0 else 0.0

# Image path resolver for fast file lookups
search_dirs = [
    str(IMAGES_DIR) if "IMAGES_DIR" in globals() else "images",
    str(REPO_ROOT / "multi_class_train_rfdetr" / "images") if "REPO_ROOT" in globals() else None,
    str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
    str(Path("coco_files/images")),
    str(Path("images")),
    "/home/jupyter/multi_class_train_rfdetr/images",
    "/home/jupyter/coco_files/images",
    "/home/jupyter/images"
]
search_dirs = [d for d in search_dirs if d and os.path.exists(d)]

def resolve_image_path(img_identifier):
    img_str = str(img_identifier)
    fname = Path(img_str).name
    stem = Path(img_str).stem
    candidates = [fname, f"{stem}.jpg", f"{img_str}.jpg", img_str]
    for d in search_dirs:
        for c in candidates:
            p = os.path.join(d, c)
            if os.path.exists(p) and os.path.getsize(p) > 0:
                return p
    return None

def compute_single_image_blur(img_identifier) -> Tuple[str, float]:
    """Computes Laplacian variance sharpness score (lower = more blurry)."""
    local_p = resolve_image_path(img_identifier)
    if not local_p:
        return img_identifier, 999.0
    try:
        try:
            import cv2
            gray = cv2.imread(local_p, cv2.IMREAD_GRAYSCALE)
            if gray is not None:
                # Downsample large images for 10x speedup
                if gray.shape[1] > 640:
                    gray = cv2.resize(gray, (640, int(gray.shape[0] * 640 / gray.shape[1])), interpolation=cv2.INTER_AREA)
                return img_identifier, float(cv2.Laplacian(gray, cv2.CV_64F).var())
        except Exception:
            pass

        im = Image.open(local_p).convert("L")
        if im.width > 640:
            scale = 640 / im.width
            im = im.resize((640, int(im.height * scale)), Image.BILINEAR)
        arr = np.array(im, dtype=np.float64)
        lap = (
            -4.0 * arr[1:-1, 1:-1]
            + arr[:-2, 1:-1]
            + arr[2:, 1:-1]
            + arr[1:-1, :-2]
            + arr[1:-1, 2:]
        )
        return img_identifier, float(lap.var())
    except Exception:
        return img_identifier, 999.0

# 1. Fast indexed lookup of raw image dimensions
raw_rec_lookup = {}
for r in raw_records:
    if isinstance(r, dict):
        cu = r.get("coco_url")
        u = r.get("url")
        if cu: raw_rec_lookup[cu] = r
        if u: raw_rec_lookup[u] = r

# 2. Precompute blur scores across images concurrently with multi-threading and tqdm
image_source = annotations_by_image if 'annotations_by_image' in globals() else image_annotations
unique_images = list(image_source.keys())

_image_blur_scores: Dict[str, float] = {}
BLUR_THRESHOLD = 80.0

logger.info(f"[CELL 5] Parallel blur analysis across {len(unique_images)} images using ThreadPoolExecutor...")
with ThreadPoolExecutor(max_workers=min(16, (os.cpu_count() or 4) * 2)) as executor:
    futures = {executor.submit(compute_single_image_blur, iid): iid for iid in unique_images}
    for fut in tqdm(as_completed(futures), total=len(unique_images), desc="[CELL 5] Blur & Sharpness Scan", unit="img"):
        try:
            iid, score = fut.result()
            _image_blur_scores[iid] = score
        except Exception:
            pass

flagged_anomalies = []
clean_annotations = []

# 3. Fast audit loop with tqdm progress bar
logger.info(f"[CELL 5] Executing 8-rule anomaly audit across {len(image_source)} image sets...")
for img_url, anns in tqdm(image_source.items(), desc="[CELL 5] Anomaly Detection Audit", unit="img"):
    # Lookup true image dimensions from registered COCO metadata or annotation dictionary
    im_meta = raw_images.get(img_url, raw_images.get(str(img_url), {})) if "raw_images" in globals() else {}
    img_w = float(im_meta.get("width") or (anns[0].get("img_w") if anns else 1920) or 1920)
    img_h = float(im_meta.get("height") or (anns[0].get("img_h") if anns else 1080) or 1080)
    
    blur_score = _image_blur_scores.get(img_url, 999.0)
    is_image_blurry = blur_score < BLUR_THRESHOLD
    
    num_anns = len(anns)
    for i, a in enumerate(anns):
        bbox = a.get("bbox", [])
        if len(bbox) != 4:
            flagged_anomalies.append({
                "annotation": a,
                "image_url": img_url,
                "reasons": ["Malformed bbox dimensions"],
                "primary_rule": "Malformed bbox dimensions",
                "bbox": [0,0,0,0],
                "category_name": a.get("category_name", "Unknown")
            })
            continue
            
        x, y, w, h = bbox[0], bbox[1], bbox[2], bbox[3]
        area = w * h
        is_anomaly = False
        reasons = []
        
        # Master check: Default treat ONLY blurry images as anomalies
        if TREAT_ONLY_BLUR_AS_ANOMALY:
            if is_image_blurry:
                is_anomaly = True
                reasons.append(f"Blurry image (Laplacian var={blur_score:.1f} < {BLUR_THRESHOLD})")
        else:
            # Full 8-rule audit mode
            if w <= 0 or h <= 0:
                is_anomaly = True
                reasons.append(f"Degenerate box (w={w}, h={h})")
            if FILTER_MICRO_DEGENERATE and (w < 2.0 or h < 2.0 or area < 4.0):
                is_anomaly = True
                reasons.append(f"Micro-click (<2px, w={w:.1f}, h={h:.1f}, area={area:.1f})")
            if w > 0.95 * img_w and h > 0.95 * img_h:
                is_anomaly = True
                reasons.append(f"Giant box covering >95% image (w={w:.1f}, h={h:.1f})")
            if is_image_blurry:
                is_anomaly = True
                reasons.append(f"Blurry image (Laplacian var={blur_score:.1f} < {BLUR_THRESHOLD})")
            if FILTER_EDGE_CROPPED_BOXES:
                is_cropped_left = x <= 1.0
                is_cropped_top = y <= 1.0
                is_cropped_right = (x + w) >= (img_w - 1.0)
                is_cropped_bottom = (y + h) >= (img_h - 1.0)
                if is_cropped_left or is_cropped_top or is_cropped_right or is_cropped_bottom:
                    border_sides = []
                    if is_cropped_left: border_sides.append("left")
                    if is_cropped_top: border_sides.append("top")
                    if is_cropped_right: border_sides.append("right")
                    if is_cropped_bottom: border_sides.append("bottom")
                    is_anomaly = True
                    reasons.append(f"Cropped box at image border ({', '.join(border_sides)})")
            if num_anns > 1:
                for j in range(num_anns):
                    if i != j and a.get("category_name") == anns[j].get("category_name"):
                        b_box = anns[j].get("bbox", [])
                        if len(b_box) == 4 and abs(x - b_box[0]) < 15 and abs(y - b_box[1]) < 15:
                            iou = compute_box_iou(bbox, b_box)
                            if iou > 0.95:
                                is_anomaly = True
                                reasons.append(f"Duplicate box with IoU {iou:.2f}")
                                break
                overlapping_count = 0
                for j in range(num_anns):
                    if i != j:
                        b_box = anns[j].get("bbox", [])
                        if len(b_box) == 4:
                            if compute_box_iou(bbox, b_box) > 0.70:
                                overlapping_count += 1
                                if overlapping_count >= 3:
                                    break
                if overlapping_count >= 3:
                    is_anomaly = True
                    reasons.append(f"Dense cluster overlap ({overlapping_count} overlapping boxes)")
            if w > 0.65 * img_w and h < 0.08 * img_h:
                is_anomaly = True
                reasons.append("Loose shelf padding across multiple bays")
            
        if is_anomaly:
            primary = reasons[0].split("(")[0].strip() if reasons else "Other anomaly"
            flagged_anomalies.append({
                "annotation": a,
                "image_url": img_url,
                "reasons": reasons,
                "primary_rule": primary,
                "bbox": bbox,
                "category_name": a.get("category_name", "Unknown")
            })
        else:
            clean_annotations.append(a)

logger.info("=" * 70)
logger.info(f"ANOMALY AUDIT COMPLETE: {len(flagged_anomalies)} / {total_boxes} boxes flagged ({len(flagged_anomalies)/max(1, total_boxes)*100:.2f}%)")
logger.info(f"Clean annotations retained: {len(clean_annotations)}")
logger.info("=" * 70)


In [ ]:
# CELL 6 - Comprehensive Anomaly Diagnostic Graphs
def normalize_anomaly_category(reason: str) -> str:
    r = str(reason).split("(")[0].strip()
    if "Duplicate box" in r:
        return "Duplicate Box Overlap"
    if "Micro-click" in r:
        return "Micro-Click (<4px)"
    if "Degenerate box" in r:
        return "Degenerate Box (Zero Dim)"
    if "Cropped box at image border" in r or "border" in r.lower():
        return "Edge-Cropped / Border-Truncated Box"
    if "Blurry" in r:
        return "Severe Image Blur"
    if "Giant box" in r:
        return "Giant Box (>95% Image)"
    if "Dense cluster" in r:
        return "Dense Cluster Overlap"
    if "Loose shelf padding" in r:
        return "Loose Shelf Padding"
    return r

rule_counts = Counter()
for item in flagged_anomalies:
    for r in item["reasons"]:
        rule_counts[normalize_anomaly_category(r)] += 1

fig, axs = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Anomaly Rule Frequency Breakdown
labels = list(rule_counts.keys()) or ["No Anomalies"]
counts = list(rule_counts.values()) or [0]
bars = axs[0, 0].barh(labels, counts, color="#E53935")
axs[0, 0].set_title("Flagged Anomalies by Rule Category", fontsize=12, fontweight="bold")
axs[0, 0].set_xlabel("Count")
for b in bars:
    axs[0, 0].text(b.get_width() + 0.5, b.get_y() + b.get_height()/2.0, str(int(b.get_width())), va="center", fontsize=10)

# Panel 2: Image Blur Score Distribution (Sharpness vs Blurry)
blur_scores_list = [s for s in _image_blur_scores.values() if s < 900.0] if "_image_blur_scores" in globals() else []
if blur_scores_list:
    axs[0, 1].hist(blur_scores_list, bins=25, color="#00897B", alpha=0.75, label="Images Inspected")
    axs[0, 1].axvline(BLUR_THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"Blur Threshold (<{BLUR_THRESHOLD})")
    axs[0, 1].set_title("Image Sharpness (Laplacian Variance Distribution)", fontsize=12, fontweight="bold")
    axs[0, 1].set_xlabel("Laplacian Variance (Sharpness Score)")
    axs[0, 1].legend()
else:
    axs[0, 1].text(0.5, 0.5, "Blur audit skipped (No local images evaluated)", ha="center", va="center", fontsize=11)
    axs[0, 1].set_title("Image Sharpness Audit", fontsize=12, fontweight="bold")

# Panel 3: Box Area (Log-scale) Comparison
clean_areas = [a["bbox"][2] * a["bbox"][3] for a in clean_annotations if len(a.get("bbox", [])) == 4 and a["bbox"][2]*a["bbox"][3] > 0]
anom_areas = [a["bbox"][2] * a["bbox"][3] for a in flagged_anomalies if len(a.get("bbox", [])) == 4 and a["bbox"][2]*a["bbox"][3] > 0]

axs[1, 0].hist(np.log10(clean_areas), bins=30, alpha=0.7, label="Clean", color="#1E88E5")
if anom_areas:
    axs[1, 0].hist(np.log10(anom_areas), bins=30, alpha=0.7, label="Anomalies", color="#E53935")
axs[1, 0].set_title("Bounding Box Area Distribution (log10 px²)", fontsize=12, fontweight="bold")
axs[1, 0].set_xlabel("log10(Area)")
axs[1, 0].legend()

# Panel 4: Quality Retention Overview Pie Chart
clean_count = len(clean_annotations)
flagged_count = len(flagged_anomalies)
if total_boxes > 0:
    axs[1, 1].pie([clean_count, flagged_count], labels=[f"Clean ({clean_count})", f"Flagged ({flagged_count})"], autopct="%1.1f%%", colors=["#43A047", "#E53935"], startangle=140, explode=(0, 0.1))
    axs[1, 1].set_title("Dataset Quality Split (Retained vs Excluded)", fontsize=12, fontweight="bold")

plt.tight_layout()
show_plt_figure(fig)


In [ ]:
# CELL 7 - Visual Display of Detected Anomaly Samples (5 Samples per Anomaly Category)
import os, sys, json
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from collections import defaultdict, Counter
from pathlib import Path
import urllib.parse

def normalize_anomaly_category(reason: str) -> str:
    """Canonical name for anomaly rule grouping."""
    r = str(reason).split("(")[0].strip()
    if "Duplicate box" in r:
        return "Duplicate Box Overlap"
    if "Micro-click" in r:
        return "Micro-Click (<4px)"
    if "Degenerate box" in r:
        return "Degenerate Box (Zero Dim)"
    if "Cropped box at image border" in r or "border" in r.lower():
        return "Edge-Cropped / Border-Truncated Box"
    if "Blurry" in r:
        return "Severe Image Blur"
    if "Giant box" in r:
        return "Giant Box (>95% Image)"
    if "Dense cluster" in r:
        return "Dense Cluster Overlap"
    if "Loose shelf padding" in r:
        return "Loose Shelf Padding"
    return r

def display_anomaly_breakdown_and_samples(
    anomalies_list, 
    images_dir: Path, 
    samples_per_anomaly: int = 5, 
    max_display_width: int = 750
):
    """
    1. Computes and displays clean counts for each flagged anomaly category.
    2. Groups anomalies by canonical category name.
    3. Displays up to samples_per_anomaly (default 5) visual image previews for EACH anomaly category.
       - Clean bounding boxes are rendered in bright green with their object tag name.
       - The target anomalous boxes are prominently highlighted in bright red/yellow with their object tag & anomaly reason.
       - Border-clipped tags display red border indicators.
    """
    print("=" * 80)
    print("[CELL 7] ANOMALY DETECTION BREAKDOWN & 5-SAMPLE PER CATEGORY INSPECTION")
    print("=" * 80)

    if not anomalies_list:
        print("Zero anomalies detected! Dataset bounding boxes passed all quality audit checks.")
        return

    # Count frequencies of each anomaly rule category
    rule_counts = Counter()
    anomalies_by_rule = defaultdict(list)
    for a in anomalies_list:
        reasons = a.get("reasons", ["Unspecified anomaly"])
        normalized_rules = {normalize_anomaly_category(r) for r in reasons}
        for nr in normalized_rules:
            rule_counts[nr] += 1
            anomalies_by_rule[nr].append(a)

    total_flagged = len(anomalies_list)
    print(f"Total Flagged Anomaly Boxes:     {total_flagged}")
    print(f"Distinct Anomaly Types Detected: {len(rule_counts)}")
    print()
    print("COUNTS OF EACH ANOMALY TYPE:")
    print("-" * 65)
    print(f"{'Anomaly Rule Category':<42} | {'Count':<8} | Share")
    print("-" * 65)
    for rule, count in rule_counts.most_common():
        pct = (count / max(1, total_flagged)) * 100
        print(f"{rule:<42} | {count:<8} | {pct:5.1f}%")
    print("-" * 65)
    print()

    # Search paths for resolving images on disk
    search_dirs = [
        str(images_dir),
        str(REPO_ROOT / "multi_class_train_rfdetr" / "images") if "REPO_ROOT" in globals() else None,
        str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
        str(Path("coco_files/images")),
        str(Path("images")),
        str(Path("dataset_stratified/images")),
        "/home/jupyter/multi_class_train_rfdetr/images",
        "/home/jupyter/coco_files/images",
        "/home/jupyter/images"
    ]
    search_dirs = [d for d in search_dirs if d and os.path.exists(d)]

    def resolve_image_path(img_identifier):
        img_str = str(img_identifier)
        fname = Path(img_str).name
        stem = Path(img_str).stem
        candidates = [fname, f"{stem}.jpg", f"{img_str}.jpg", img_str]
        for d in search_dirs:
            for c in candidates:
                p = os.path.join(d, c)
                if os.path.exists(p) and os.path.getsize(p) > 0:
                    return p
        return None

    img_ann_lookup = annotations_by_image if "annotations_by_image" in globals() else image_annotations
    globally_displayed_images = set()

    # Distinct colors for drawing
    COLOR_CLEAN = (0, 200, 83)        # Vibrant green
    COLOR_TARGET_ANOM = (229, 57, 53)  # Bright red
    COLOR_OTHER_ANOM = (255, 149, 0)   # Orange
    COLOR_BADGE_BG = (255, 214, 0)     # Yellow
    COLOR_BADGE_TEXT = (0, 0, 0)       # Black

    # Render exactly samples_per_anomaly for each detected anomaly rule
    for rule, items in tqdm(rule_counts.most_common(), desc="[CELL 7] Inspecting Anomaly Types", unit="rule"):
        rule_items = anomalies_by_rule[rule]
        
        # Collect distinct images for this rule
        images_for_this_rule = []
        seen_imgs_local = set()
        for a in rule_items:
            u = a.get("image_url")
            if u and u not in seen_imgs_local:
                seen_imgs_local.add(u)
                images_for_this_rule.append(u)

        fresh_images = [u for u in images_for_this_rule if u not in globally_displayed_images]
        fallback_images = [u for u in images_for_this_rule if u in globally_displayed_images]
        ordered_images = fresh_images + fallback_images

        print("=" * 80)
        print(f">>> ANOMALY CATEGORY: [{rule.upper()}]")
        print(f"    Total Anomaly Boxes: {len(rule_items)} across {len(ordered_images)} image(s)")
        print(f"    Showing up to {samples_per_anomaly} sample images with category tags & anomaly highlights")
        print("=" * 80)

        rendered_count = 0

        for img_url in ordered_images:
            if rendered_count >= samples_per_anomaly:
                break

            local_path = resolve_image_path(img_url)
            pil_img = None

            if local_path and os.path.exists(local_path):
                try:
                    pil_img = Image.open(local_path).convert("RGB")
                    fname = Path(local_path).name
                except Exception as e:
                    pil_img = None

            # Fallback: If image url is HTTP/HTTPS, attempt remote fetch
            if pil_img is None and (str(img_url).startswith("http://") or str(img_url).startswith("https://")):
                try:
                    import requests, io
                    resp = requests.get(str(img_url), timeout=4)
                    if resp.status_code == 200:
                        pil_img = Image.open(io.BytesIO(resp.content)).convert("RGB")
                        fname = Path(urllib.parse.urlparse(str(img_url)).path).name or "remote_sample.jpg"
                except Exception:
                    pil_img = None

            # Synthetic Canvas Fallback: if file is not locally on disk, render high-detail canvas with image resolution
            if pil_img is None:
                img_record = raw_rec_lookup.get(img_url, {}) if "raw_rec_lookup" in globals() else {}
                w_fallback = int(img_record.get("width", 1920) or 1920)
                h_fallback = int(img_record.get("height", 1080) or 1080)
                pil_img = Image.new("RGB", (w_fallback, h_fallback), color=(245, 245, 247))
                d_synth = ImageDraw.Draw(pil_img)
                fname = Path(str(img_url)).name
                # Draw subtle watermark grid
                d_synth.rectangle([2, 2, w_fallback - 3, h_fallback - 3], outline=(200, 200, 200), width=2)
                d_synth.text((20, 20), f"[OFFLINE PREVIEW CANVAS] {fname} ({w_fallback}x{h_fallback})", fill=(120, 120, 120))

            globally_displayed_images.add(img_url)
            im_w, im_h = pil_img.size
            draw = ImageDraw.Draw(pil_img)

            # Partition annotations in this image into:
            # 1. Target anomalies (matching this rule)
            # 2. Other anomalies (flagged for different rules)
            # 3. Clean annotations
            all_boxes_in_img = img_ann_lookup.get(img_url, [])
            
            anoms_in_img = [a for a in anomalies_list if a.get("image_url") == img_url]
            matching_anoms = [
                a for a in anoms_in_img 
                if any(normalize_anomaly_category(r) == rule for r in a.get("reasons", []))
            ]
            matching_bbox_tuples = [tuple(a.get("bbox", [])) for a in matching_anoms]

            other_anoms = [
                a for a in anoms_in_img 
                if tuple(a.get("bbox", [])) not in matching_bbox_tuples
            ]
            other_bbox_tuples = [tuple(a.get("bbox", [])) for a in other_anoms]

            # 1. Draw CLEAN annotations in green with category tag badge
            for c in all_boxes_in_img:
                cb = c.get("bbox", [])
                if len(cb) == 4 and tuple(cb) not in matching_bbox_tuples and tuple(cb) not in other_bbox_tuples:
                    x0 = max(0.0, min(float(im_w - 1), float(cb[0])))
                    y0 = max(0.0, min(float(im_h - 1), float(cb[1])))
                    x1 = max(0.0, min(float(im_w - 1), float(cb[0] + cb[2])))
                    y1 = max(0.0, min(float(im_h - 1), float(cb[1] + cb[3])))
                    draw.rectangle([x0, y0, x1, y1], outline=COLOR_CLEAN, width=2)

                    # Green tag badge with category name
                    tag_name = c.get("category_name", "Tag")
                    label_str = f" {tag_name} "
                    if hasattr(draw, "textbbox"):
                        tb = draw.textbbox((x0, max(0.0, y0 - 15.0)), label_str)
                        draw.rectangle(tb, fill=COLOR_CLEAN)
                        draw.text((x0, max(0.0, y0 - 15.0)), label_str, fill=(255, 255, 255))
                    else:
                        draw.text((x0 + 2, max(0.0, y0 - 13.0)), label_str, fill=COLOR_CLEAN)

            # 2. Draw OTHER anomalies (if any) in orange
            for o_item in other_anoms:
                ob = o_item.get("bbox", [0, 0, 0, 0])
                if len(ob) != 4:
                    continue
                ox0 = max(0.0, min(float(im_w - 1), float(ob[0])))
                oy0 = max(0.0, min(float(im_h - 1), float(ob[1])))
                ox1 = max(0.0, min(float(im_w - 1), float(ob[0] + max(1.0, ob[2]))))
                oy1 = max(0.0, min(float(im_h - 1), float(ob[1] + max(1.0, ob[3]))))
                draw.rectangle([ox0, oy0, ox1, oy1], outline=COLOR_OTHER_ANOM, width=2)
                
                o_tag = o_item.get("category_name", "Tag")
                o_label = f" ! {o_tag} "
                if hasattr(draw, "textbbox"):
                    tb = draw.textbbox((ox0, max(0.0, oy0 - 15.0)), o_label)
                    draw.rectangle(tb, fill=COLOR_OTHER_ANOM)
                    draw.text((ox0, max(0.0, oy0 - 15.0)), o_label, fill=(255, 255, 255))

            # 3. Draw TARGET RULE ANOMALIES in bold red with prominent yellow callout badge
            for a_item in matching_anoms:
                ab = a_item.get("bbox", [0, 0, 0, 0])
                if len(ab) != 4:
                    continue
                ax0 = max(0.0, min(float(im_w - 1), float(ab[0])))
                ay0 = max(0.0, min(float(im_h - 1), float(ab[1])))
                ax1 = max(0.0, min(float(im_w - 1), float(ab[0] + max(1.0, ab[2]))))
                ay1 = max(0.0, min(float(im_h - 1), float(ab[1] + max(1.0, ab[3]))))

                # Red highlight box (heavy border)
                draw.rectangle([ax0, ay0, ax1, ay1], outline=COLOR_TARGET_ANOM, width=4)

                # Edge indicator: if cropped at border, draw prominent border line along touching edge
                if "border" in rule.lower() or "cropped" in rule.lower():
                    if ax0 <= 1.0:
                        draw.line([(0, ay0), (0, ay1)], fill=(255, 0, 0), width=8)
                    if ay0 <= 1.0:
                        draw.line([(ax0, 0), (ax1, 0)], fill=(255, 0, 0), width=8)
                    if ax1 >= im_w - 2.0:
                        draw.line([(im_w - 1, ay0), (im_w - 1, ay1)], fill=(255, 0, 0), width=8)
                    if ay1 >= im_h - 2.0:
                        draw.line([(ax0, im_h - 1), (ax1, im_h - 1)], fill=(255, 0, 0), width=8)

                # Prominent callout badge showing tag name + anomaly reason
                cat_name = a_item.get("category_name", "Tag")
                specific_reason = a_item.get("reasons", [rule])[0]
                badge_text = f" ! [{cat_name}] ANOMALY: {specific_reason} "

                if hasattr(draw, "textbbox"):
                    tb = draw.textbbox((ax0, max(0.0, ay0 - 18.0)), badge_text)
                    # Yellow badge backing
                    draw.rectangle(tb, fill=COLOR_BADGE_BG)
                    draw.rectangle(tb, outline=COLOR_TARGET_ANOM, width=1)
                    draw.text((ax0, max(0.0, ay0 - 18.0)), badge_text, fill=COLOR_BADGE_TEXT)
                else:
                    draw.text((ax0 + 2, max(0.0, ay0 - 14.0)), badge_text, fill=COLOR_TARGET_ANOM)

            rendered_count += 1
            matching_tags = [a.get("category_name", "Unknown") for a in matching_anoms]
            print(f"\n[Sample {rendered_count}/{samples_per_anomaly}] Category: [{rule}]")
            print(f"  • Image File:    {fname} ({im_w}x{im_h})")
            print(f"  • Target Tag(s): {', '.join(set(matching_tags))} ({len(matching_anoms)} anomaly boxes)")
            print(f"  • Reasons:       {' | '.join(matching_anoms[0].get('reasons', []))}")

            # Scale for notebook inline display
            if im_w > max_display_width:
                scale = max_display_width / im_w
                display_img = pil_img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
            else:
                display_img = pil_img

            display(display_img)

        if rendered_count == 0:
            print(f"[INFO] No candidate images could be retrieved for rule '{rule}'.")

    print("\n" + "=" * 80)
    print("✓ Anomaly sample inspection complete across all categories.")
    print("=" * 80)

# Execute the 5-sample per anomaly inspector
samples_to_show = globals().get("SHOW_ANOMALY_SAMPLES_PER_CATEGORY", 5)
display_anomaly_breakdown_and_samples(flagged_anomalies, IMAGES_DIR, samples_per_anomaly=samples_to_show)


In [ ]:
# CELL 8 - Export Manifest & Sanitized Dataset
manifest_path = OUTPUT_ANOMALIES_DIR / "anomalies_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "flagged_count": len(flagged_anomalies),
        "total_audited": total_boxes,
        "clean_count": len(clean_annotations),
        "anomalies": flagged_anomalies
    }, f, indent=2)

logger.info(f"✓ Saved anomalies manifest: {manifest_path}")

clean_coco = {
    "images": list({a.get("image_id"): {
        "id": a.get("image_id"),
        "file_name": a.get("file_name", f"{a.get('image_id')}.jpg"),
        "width": int(a.get("img_w", 1920)),
        "height": int(a.get("img_h", 1080)),
        "coco_url": a.get("image_url")
    } for a in clean_annotations}.values()),
    "annotations": clean_annotations,
    "categories": categories_list
}
clean_json_path = CLEANED_DATASET_DIR / "_annotations.coco.json"
with open(clean_json_path, "w", encoding="utf-8") as f:
    json.dump(clean_coco, f, indent=2)

logger.info(f"✓ Saved sanitized dataset: {clean_json_path} ({len(clean_coco['annotations'])} clean annotations)")


In [ ]:
# CELL 9 - Step B Verification on Cleaned Dataset
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Verify sanitized output file exists
target_cleaned_ann = clean_json_path if 'clean_json_path' in globals() else CLEANED_DATASET_DIR / "_annotations.coco.json"
if target_cleaned_ann.exists():
    cleaned_info = load_coco_info(str(target_cleaned_ann))
    logger.info("=" * 65)
    logger.info(f"Cleaned Dataset Audit: {cleaned_info['num_images']} images, {cleaned_info['num_anns']} annotations, {cleaned_info['num_classes']} classes")
    logger.info(f"Classes: {cleaned_info['categories']}")
    logger.info("=" * 65)
else:
    logger.info(f"Cleaned dataset file {target_cleaned_ann} not generated yet.")


In [ ]:
import json, os, sys
from pathlib import Path
# CELL 10 - Anomaly Selection & Pipeline Export Configuration (Gate to Step 03)
# ==============================================================================
# PIPELINE CONTROL TOGGLE:
# EXCLUDE_ANOMALIES_FROM_TRAINING:
#   - True  (Recommended): Exclude all selected anomalies from downstream training.
#                          Sanitizes the training dataset and exports cleaned annotations.
#   - False (Audit Only): Retain all annotations for training while logging anomalies.
#
# ACTIVE ANOMALY RULES CONFIGURATION:
# Set True / False to include or exclude specific anomaly types from being filtered:
# ==============================================================================

# ==============================================================================
# MASTER ANOMALY SELECTION TOGGLES
# ==============================================================================
EXCLUDE_ANOMALIES_FROM_TRAINING = True
FILTER_EDGE_CROPPED_BOXES = globals().get('FILTER_EDGE_CROPPED_BOXES', False)
FILTER_MICRO_DEGENERATE = globals().get('FILTER_MICRO_DEGENERATE', False)

CONSIDER_ANOMALIES = {
    'Blurry image': True,                                       # EXCLUDE ONLY: Default treat ONLY blurry images as anomalies (User requirement)
    'Cropped box at image border': False,                       # KEEP in training: User verified border tags are legitimate valid annotations
    'Micro-click': False,                                       # KEEP in training: Small/distant shelf tags are legitimate valid annotations
    'Degenerate box': False,                                    # KEEP in training
    'Giant box covering >95% image': False,                     # KEEP in training
    'Duplicate box with IoU': False,                            # KEEP in training
    'Dense cluster overlap': False,                             # KEEP in training: Warehouse shelves naturally have dense tag clusters
    'Loose shelf padding': False,                               # KEEP in training
    'Malformed bbox dimensions': False                          # KEEP in training
}

print('=' * 80)
print('[CELL 10] ANOMALY SELECTION & PIPELINE EXPORT CONFIGURATION')
print('=' * 80)
print(f'Mode: {"EXCLUDE ANOMALIES FROM TRAINING" if EXCLUDE_ANOMALIES_FROM_TRAINING else "AUDIT ONLY (Retain all annotations)"}')
print()
print('Active Anomaly Filters:')
for rule_name, active in CONSIDER_ANOMALIES.items():
    status_str = 'EXCLUDE from training' if active else 'KEEP in training'
    mark = 'X' if active else ' '
    print(f'  [{mark}] {rule_name:<35} -> {status_str}')
print('-' * 80)

# Filter anomalies according to CONSIDER_ANOMALIES configuration
final_excluded_anomalies = []
final_retained_annotations = []

for anom in flagged_anomalies:
    # Check if this anomaly matches any active rule
    matched_active_rule = False
    for r in anom.get('reasons', []):
        r_key = r.split('(')[0].strip()
        if CONSIDER_ANOMALIES.get(r_key, True):
            matched_active_rule = True
            break
            
    if matched_active_rule and EXCLUDE_ANOMALIES_FROM_TRAINING:
        final_excluded_anomalies.append(anom)
    else:
        # Re-admit into clean training annotations if rule was disabled or mode is Audit Only
        final_retained_annotations.append(anom.get('annotation', anom))

# Add base clean annotations
final_retained_annotations.extend(clean_annotations)

retained_pct = (len(final_retained_annotations) / max(1, total_boxes)) * 100
excluded_pct = (len(final_excluded_anomalies) / max(1, total_boxes)) * 100
print(f'Final Training Box Count: {len(final_retained_annotations)} / {total_boxes} ({retained_pct:.1f}% retained)')
print(f'Excluded Anomaly Boxes:   {len(final_excluded_anomalies)} ({excluded_pct:.1f}% excluded)')

# Export the active training dataset for Step 03
final_training_coco = {
    'info': {
        'description': f'RF-DETR Multi-Class Training Dataset ({"Cleaned" if EXCLUDE_ANOMALIES_FROM_TRAINING else "Full Audit"})',
        'exclude_anomalies': EXCLUDE_ANOMALIES_FROM_TRAINING,
        'active_rules': CONSIDER_ANOMALIES
    },
    'images': list({
        a.get('image_id', idx): {
            'id': a.get('image_id', idx),
            'file_name': a.get('file_name') or (raw_images.get(a.get('image_id'), {}).get('file_name') if 'raw_images' in globals() else None) or f"{a.get('image_id', idx)}.jpg",
            'width': int(a.get('img_w') or (raw_images.get(a.get('image_id'), {}).get('width', 1920))),
            'height': int(a.get('img_h') or (raw_images.get(a.get('image_id'), {}).get('height', 1080))),
            'coco_url': a.get('image_url') or a.get('file_name')
        } for idx, a in enumerate(final_retained_annotations)
    }.values()),
    'annotations': [
        {
            'id': a.get('id', idx + 1),
            'image_id': a.get('image_id', 0),
            'category_id': a.get('category_id', 0),
            'category_name': a.get('category_name', 'location_tag'),
            'bbox': a.get('bbox', [0, 0, 0, 0]),
            'area': a.get('area', 0.0),
            'iscrowd': 0,
            'segmentation': a.get('segmentation', [])
        }
        for idx, a in enumerate(final_retained_annotations)
    ],
    'categories': categories_list
}

training_export_path = CLEANED_DATASET_DIR / '_annotations.coco.json'
with open(training_export_path, 'w', encoding='utf-8') as f:
    json.dump(final_training_coco, f, indent=2)

# Also mirror to REPO_ROOT / cleaned_dataset so Step 03 finds it regardless of current working directory
mirror_export_path = REPO_ROOT / 'cleaned_dataset' / '_annotations.coco.json'
mirror_export_path.parent.mkdir(parents=True, exist_ok=True)
with open(mirror_export_path, 'w', encoding='utf-8') as f:
    json.dump(final_training_coco, f, indent=2)

# Save updated manifest
manifest_export_path = OUTPUT_ANOMALIES_DIR / 'anomalies_manifest.json'
with open(manifest_export_path, 'w', encoding='utf-8') as f:
    json.dump({
        'exclude_anomalies_from_training': EXCLUDE_ANOMALIES_FROM_TRAINING,
        'consider_anomalies_rules': CONSIDER_ANOMALIES,
        'total_audited': total_boxes,
        'clean_retained_count': len(final_retained_annotations),
        'excluded_anomalies_count': len(final_excluded_anomalies),
        'excluded_anomalies': final_excluded_anomalies
    }, f, indent=2)

print()
print(f'Saved training dataset: {training_export_path}')
print(f'Saved updated manifest: {manifest_export_path}')
print('=' * 80)
print('PROCEED TO STEP 03: Run 03_stratified_train_val_test_split.ipynb to generate stratified splits.')
print('=' * 80)
